# Corrected Global Baseline Price Model — CA and US

This notebook implements the leakage-free baseline agreed in the modeling plan:

```text
Forward7DayASP = w30 × ProductASP30
               + w90 × ProductASP90
               + wf  × FamilyAdjustedPrice
```

The target is the product's unit-weighted ASP over the next seven calendar days. Every feature uses only information available before the scoring date.

`FamilyAdjustedPrice` imports peer price movement without directly averaging products at different price levels:

```text
FamilyAdjustedPrice = ProductASP90 × (PeerASP30 / PeerASP90)
```

The peer calculation excludes the target product. CA and US are modeled separately, but weights are global within each country and are not split by family.

In [1]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

ROOT = Path.cwd()
if not (ROOT / 'Top 30 Product for US-CA').exists():
    ROOT = Path('baseline model_Brynn')
DATA_DIR = ROOT / 'Top 30 Product for US-CA'
ARTIFACT_DIR = ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
FILES = {country: DATA_DIR / f'{country}_selected_top30_transactions.csv' for country in ['CA', 'US']}

FEATURES = ['ASP30', 'ASP90', 'FamilyAdjustedPrice']
WEIGHT_NAMES = ['w30', 'w90', 'wf']

## 1. Build the daily panel

Transaction rows are aggregated using `sum(revenue) / sum(units)`. Missing-sales days retain zero units and revenue; ASP is never filled with zero. Product and peer rolling windows exclude the scoring date. The family peer signal is calculated from other Top 30 products in the same family.

In [2]:
def build_daily_panel(path):
    transactions = pd.read_csv(path, low_memory=False)
    transactions['date'] = pd.to_datetime(transactions['pricing_date'], errors='coerce')
    for column in ['revenue', 'order_units']:
        transactions[column] = pd.to_numeric(transactions[column], errors='coerce')
    transactions = transactions.loc[
        transactions['date'].notna() & transactions['revenue'].notna() &
        transactions['order_units'].notna() & (transactions['revenue'] > 0) &
        (transactions['order_units'] > 0)
    ].copy()

    family_map = transactions.groupby('product_id')['product_family'].agg(
        lambda values: values.dropna().mode().iloc[0] if len(values.dropna()) else '<missing>'
    )
    daily_sales = transactions.groupby(['product_id', 'date'], as_index=False)[['revenue', 'order_units']].sum()

    products = sorted(daily_sales['product_id'].unique())
    dates = pd.date_range(daily_sales['date'].min(), daily_sales['date'].max(), freq='D')
    complete_index = pd.MultiIndex.from_product([products, dates], names=['product_id', 'date'])
    panel = daily_sales.set_index(['product_id', 'date']).reindex(complete_index, fill_value=0).reset_index()
    panel['product_family'] = panel['product_id'].map(family_map)

    family_totals = panel.groupby(['product_family', 'date'])[['revenue', 'order_units']].transform('sum')
    panel['peer_revenue'] = family_totals['revenue'] - panel['revenue']
    panel['peer_units'] = family_totals['order_units'] - panel['order_units']

    product_panels = []
    for _, product in panel.groupby('product_id', sort=False):
        product = product.sort_values('date').copy()
        for window in [30, 90]:
            product[f'revenue_{window}d'] = product['revenue'].rolling(window, min_periods=1).sum().shift(1)
            product[f'units_{window}d'] = product['order_units'].rolling(window, min_periods=1).sum().shift(1)
            product[f'ASP{window}'] = product[f'revenue_{window}d'] / product[f'units_{window}d']

            peer_revenue = product['peer_revenue'].rolling(window, min_periods=1).sum().shift(1)
            peer_units = product['peer_units'].rolling(window, min_periods=1).sum().shift(1)
            product[f'PeerASP{window}'] = peer_revenue / peer_units

        product['FamilyAdjustedPrice'] = product['ASP90'] * (product['PeerASP30'] / product['PeerASP90'])

        # Target includes the scoring date and the following six calendar days.
        product['forward_revenue_7d'] = product['revenue'].iloc[::-1].rolling(7, min_periods=1).sum().iloc[::-1].to_numpy()
        product['forward_units_7d'] = product['order_units'].iloc[::-1].rolling(7, min_periods=1).sum().iloc[::-1].to_numpy()
        product['forward_asp_7d'] = product['forward_revenue_7d'] / product['forward_units_7d']
        product_panels.append(product)

    panel = pd.concat(product_panels, ignore_index=True)
    panel = panel.dropna(subset=FEATURES + ['forward_asp_7d'])
    panel = panel.loc[panel['forward_units_7d'] > 0].copy()
    return panel.sort_values(['date', 'product_id']).reset_index(drop=True)

panels = {country: build_daily_panel(path) for country, path in FILES.items()}
pd.DataFrame([
    {'country': country, 'rows': len(panel), 'products': panel.product_id.nunique(),
     'start': panel.date.min().date(), 'end': panel.date.max().date()}
    for country, panel in panels.items()
])

,country,rows,products,start,end
0,CA,8570,29,2023-07-19,2026-08-29
1,US,14846,30,2023-05-22,2026-08-29


## 2. Objectives and candidate formulas

Three constrained static-weight objectives are tested:

- unweighted MAE;
- unit-weighted MAE;
- balanced MAE = 50% unweighted MAE + 50% unit-weighted MAE.

All static weights are nonnegative and sum to one. A dynamic shrinkage candidate is also tested, but it is accepted only if it consistently improves both rolling validation periods.

The official selection metric is balanced MAE. This prevents a few large-volume observations from controlling the result while still reflecting business volume.

In [3]:
def metric_values(data, prediction):
    actual = data['forward_asp_7d'].to_numpy()
    units = data['forward_units_7d'].to_numpy()
    absolute_error = np.abs(actual - prediction)
    mae = absolute_error.mean()
    unit_weighted_mae = np.average(absolute_error, weights=units)
    balanced_mae = 0.5 * mae + 0.5 * unit_weighted_mae
    unit_weighted_rmse = np.sqrt(np.average((actual - prediction) ** 2, weights=units))
    weighted_mean = np.average(actual, weights=units)
    r2 = 1 - np.sum(units * (actual - prediction) ** 2) / np.sum(units * (actual - weighted_mean) ** 2)
    return {
        'MAE': mae,
        'unit_weighted_MAE': unit_weighted_mae,
        'balanced_MAE': balanced_mae,
        'unit_weighted_RMSE': unit_weighted_rmse,
        'R2': r2,
    }

def fit_static_weights(data, objective_name):
    X = data[FEATURES].to_numpy()
    actual = data['forward_asp_7d'].to_numpy()
    units = data['forward_units_7d'].to_numpy()

    def objective(weights):
        absolute_error = np.abs(actual - X @ weights)
        mae = absolute_error.mean()
        weighted_mae = np.average(absolute_error, weights=units)
        if objective_name == 'MAE':
            return mae
        if objective_name == 'UnitWeightedMAE':
            return weighted_mae
        return 0.5 * mae + 0.5 * weighted_mae

    result = minimize(
        objective,
        x0=np.ones(3) / 3,
        method='SLSQP',
        bounds=[(0, 1)] * 3,
        constraints={'type': 'eq', 'fun': lambda weights: weights.sum() - 1},
        options={'maxiter': 1000, 'ftol': 1e-12},
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result.x

def fit_dynamic_shrinkage(data):
    actual = data['forward_asp_7d'].to_numpy()
    units = data['forward_units_7d'].to_numpy()
    recent_units = data['units_30d'].to_numpy()
    asp30 = data['ASP30'].to_numpy()
    asp90 = data['ASP90'].to_numpy()
    family = data['FamilyAdjustedPrice'].to_numpy()

    def objective(parameters):
        k = np.exp(parameters[0])
        alpha = 1 / (1 + np.exp(-parameters[1]))
        recent_weight = recent_units / (recent_units + k)
        prediction = recent_weight * asp30 + (1 - recent_weight) * (alpha * asp90 + (1 - alpha) * family)
        return metric_values(data, prediction)['balanced_MAE']

    result = minimize(objective, x0=[np.log(10), 0], method='Nelder-Mead', options={'maxiter': 2000})
    return {'k': np.exp(result.x[0]), 'alpha': 1 / (1 + np.exp(-result.x[1]))}

def predict_model(data, model_name, parameters=None):
    if model_name == 'ASP30_only':
        return data['ASP30'].to_numpy()
    if model_name == 'ASP90_only':
        return data['ASP90'].to_numpy()
    if model_name == 'Fixed_50_30_20':
        return data[FEATURES].to_numpy() @ np.array([0.5, 0.3, 0.2])
    if model_name.startswith('Learned_'):
        return data[FEATURES].to_numpy() @ np.asarray(parameters)
    if model_name == 'Dynamic_shrinkage':
        recent_weight = data['units_30d'].to_numpy() / (data['units_30d'].to_numpy() + parameters['k'])
        fallback = parameters['alpha'] * data['ASP90'].to_numpy() + (1 - parameters['alpha']) * data['FamilyAdjustedPrice'].to_numpy()
        return recent_weight * data['ASP30'].to_numpy() + (1 - recent_weight) * fallback
    raise ValueError(model_name)

## 3. Rolling fiscal-period validation

The same calendar boundaries are used for all products:

- Fold 1: train through FY25Q4; validate on FY26Q1–FY26Q2.
- Fold 2: train through FY26Q2; validate on FY26Q3–FY26Q4.
- Final test: train through FY26Q4; test on FY27Q1.
- FY27Q2 is incomplete and is reported only as a shadow evaluation.

A learned formula is eligible only when it beats ASP30-only in both validation folds and improves average balanced MAE by at least 2%. Otherwise the official baseline falls back to ASP30-only.

In [4]:
FOLDS = {
    'fold_1': {'train_end': '2025-03-31', 'validation_start': '2025-04-01', 'validation_end': '2025-10-05'},
    'fold_2': {'train_end': '2025-10-05', 'validation_start': '2025-10-06', 'validation_end': '2026-03-31'},
}
STATIC_OBJECTIVES = {
    'Learned_MAE': 'MAE',
    'Learned_UnitWeightedMAE': 'UnitWeightedMAE',
    'Learned_BalancedMAE': 'BalancedMAE',
}

backtest_rows = []
weight_rows = []
for country, panel in panels.items():
    for fold_name, dates in FOLDS.items():
        train = panel[panel['date'] <= dates['train_end']]
        validation = panel[(panel['date'] >= dates['validation_start']) & (panel['date'] <= dates['validation_end'])]

        models = {
            'ASP30_only': None,
            'ASP90_only': None,
            'Fixed_50_30_20': np.array([0.5, 0.3, 0.2]),
        }
        for model_name, objective_name in STATIC_OBJECTIVES.items():
            models[model_name] = fit_static_weights(train, objective_name)
        models['Dynamic_shrinkage'] = fit_dynamic_shrinkage(train)

        for model_name, parameters in models.items():
            prediction = predict_model(validation, model_name, parameters)
            backtest_rows.append({
                'country': country, 'fold': fold_name, 'model': model_name,
                'train_rows': len(train), 'validation_rows': len(validation),
                **metric_values(validation, prediction),
            })
            if model_name.startswith('Learned_'):
                weight_rows.append({
                    'country': country, 'fold': fold_name, 'model': model_name,
                    'w30': parameters[0], 'w90': parameters[1], 'wf': parameters[2],
                })
            elif model_name == 'Dynamic_shrinkage':
                weight_rows.append({
                    'country': country, 'fold': fold_name, 'model': model_name,
                    'k': parameters['k'], 'alpha': parameters['alpha'],
                })

backtest_results = pd.DataFrame(backtest_rows)
weight_stability = pd.DataFrame(weight_rows)
backtest_results.to_csv(ARTIFACT_DIR / 'corrected_backtest_comparison.csv', index=False)
weight_stability.to_csv(ARTIFACT_DIR / 'corrected_weight_stability.csv', index=False)

backtest_results.sort_values(['country', 'fold', 'balanced_MAE'])

,country,fold,model,train_rows,validation_rows,MAE,unit_weighted_MAE,balanced_MAE,unit_weighted_RMSE,R2
0,CA,fold_1,ASP30_only,1635,2885,137.9719,138.1523,138.0621,269.9899,0.8301
6,CA,fold_1,Dynamic_shrinkage,1635,2885,137.9719,138.1523,138.0621,269.9899,0.8301
2,CA,fold_1,Fixed_50_30_20,1635,2885,135.3542,141.0389,138.1965,278.0301,0.8198
4,CA,fold_1,Learned_UnitWeightedMAE,1635,2885,136.7824,140.1430,138.4627,274.9999,0.8237
5,CA,fold_1,Learned_BalancedMAE,1635,2885,137.3303,141.1964,139.2633,276.5935,0.8216
3,CA,fold_1,Learned_MAE,1635,2885,137.3303,141.1964,139.2633,276.5936,0.8216
1,CA,fold_1,ASP90_only,1635,2885,139.7517,155.1273,147.4395,304.4397,0.7839
7,CA,fold_2,ASP30_only,4520,2724,117.5677,110.6391,114.1034,177.0108,0.8980
9,CA,fold_2,Fixed_50_30_20,4520,2724,127.3211,121.3828,124.3519,186.8018,0.8865
11,CA,fold_2,Learned_UnitWeightedMAE,4520,2724,128.1301,126.9568,127.5435,197.6156,0.8729


In [5]:
selection_rows = []
for country in panels:
    country_results = backtest_results.query('country == @country')
    asp30 = country_results.query("model == 'ASP30_only'").set_index('fold')['balanced_MAE']
    for model_name in country_results['model'].unique():
        result = country_results.query('model == @model_name').set_index('fold')['balanced_MAE']
        improvements = 1 - result / asp30
        eligible = bool((improvements > 0).all() and improvements.mean() >= 0.02)
        selection_rows.append({
            'country': country,
            'model': model_name,
            'mean_balanced_MAE': result.mean(),
            'mean_improvement_vs_ASP30': improvements.mean(),
            'improved_both_folds': bool((improvements > 0).all()),
            'eligible': eligible if model_name != 'ASP30_only' else True,
        })

selection_table = pd.DataFrame(selection_rows)
selected_models = {}
for country in panels:
    candidates = selection_table.query('country == @country and eligible')
    selected_models[country] = candidates.sort_values('mean_balanced_MAE').iloc[0]['model']

selection_table.to_csv(ARTIFACT_DIR / 'corrected_model_selection.csv', index=False)
selection_table.sort_values(['country', 'mean_balanced_MAE'])

,country,model,mean_balanced_MAE,mean_improvement_vs_ASP30,improved_both_folds,eligible
0,CA,ASP30_only,126.0828,0.0000,False,True
2,CA,Fixed_50_30_20,131.2742,-0.0454,False,False
4,CA,Learned_UnitWeightedMAE,133.0031,-0.0603,False,False
5,CA,Learned_BalancedMAE,133.9130,-0.0677,False,False
3,CA,Learned_MAE,133.9757,-0.0683,False,False
6,CA,Dynamic_shrinkage,135.4749,-0.0823,False,False
1,CA,ASP90_only,141.5829,-0.1287,False,False
13,US,Dynamic_shrinkage,110.5793,0.0128,False,False
7,US,ASP30_only,112.1729,0.0000,False,True
12,US,Learned_BalancedMAE,112.2613,-0.0013,False,False


## 4. Final FY27Q1 test and FY27Q2 shadow evaluation

The selected method is refitted using all data through FY26Q4. The FY27Q1 test set is untouched during selection. FY27Q2 is incomplete and is not used as the official test result.

In [6]:
final_rows = []
prediction_outputs = []
for country, panel in panels.items():
    model_name = selected_models[country]
    train = panel[panel['date'] <= '2026-03-31']
    test = panel[(panel['date'] >= '2026-04-01') & (panel['date'] <= '2026-06-30')]
    shadow = panel[panel['date'] >= '2026-07-01']

    if model_name in STATIC_OBJECTIVES:
        parameters = fit_static_weights(train, STATIC_OBJECTIVES[model_name])
        parameter_record = {'w30': parameters[0], 'w90': parameters[1], 'wf': parameters[2]}
    elif model_name == 'Dynamic_shrinkage':
        parameters = fit_dynamic_shrinkage(train)
        parameter_record = {'k': parameters['k'], 'alpha': parameters['alpha']}
    elif model_name == 'ASP30_only':
        parameters = None
        parameter_record = {'w30': 1.0, 'w90': 0.0, 'wf': 0.0}
    elif model_name == 'ASP90_only':
        parameters = None
        parameter_record = {'w30': 0.0, 'w90': 1.0, 'wf': 0.0}
    else:
        parameters = np.array([0.5, 0.3, 0.2])
        parameter_record = {'w30': 0.5, 'w90': 0.3, 'wf': 0.2}

    for period_name, evaluation_data in [('FY27Q1_test', test), ('FY27Q2_shadow', shadow)]:
        prediction = predict_model(evaluation_data, model_name, parameters)
        final_rows.append({
            'country': country, 'selected_model': model_name, 'period': period_name,
            'rows': len(evaluation_data), **parameter_record,
            **metric_values(evaluation_data, prediction),
        })
        output = evaluation_data[['date', 'product_id', 'product_family', 'forward_asp_7d',
                                  'forward_units_7d', 'ASP30', 'ASP90', 'FamilyAdjustedPrice']].copy()
        output['predicted_price'] = prediction
        output['country'] = country
        output['period'] = period_name
        output['selected_model'] = model_name
        prediction_outputs.append(output)

final_results = pd.DataFrame(final_rows)
final_predictions = pd.concat(prediction_outputs, ignore_index=True)
final_results.to_csv(ARTIFACT_DIR / 'corrected_final_model_summary.csv', index=False)
final_predictions.to_csv(ARTIFACT_DIR / 'corrected_final_predictions.csv', index=False)
final_results

,country,selected_model,period,rows,w30,w90,wf,MAE,unit_weighted_MAE,balanced_MAE,unit_weighted_RMSE,R2
0,CA,ASP30_only,FY27Q1_test,322,1.0000,0.0000,0.0000,297.8690,368.9521,333.4105,529.4706,0.2542
1,CA,ASP30_only,FY27Q2_shadow,1004,1.0000,0.0000,0.0000,161.3077,102.2120,131.7599,203.8975,0.7926
2,US,ASP30_only,FY27Q1_test,516,1.0000,0.0000,0.0000,243.9530,266.2742,255.1136,394.9843,0.6321
3,US,ASP30_only,FY27Q2_shadow,1011,1.0000,0.0000,0.0000,157.7930,102.5961,130.1945,161.4277,0.8993


## 5. Final conclusion

The next cell prints the accepted country-level formula. A `1/0/0` result is an intentional fallback to ASP30-only when a more complex blend does not deliver stable validation improvement. This is preferable to publishing unstable coefficients.

In [7]:
conclusion = {}
for country in ['CA', 'US']:
    row = final_results.query("country == @country and period == 'FY27Q1_test'").iloc[0]
    conclusion[country] = {
        'selected_model': row['selected_model'],
        'formula': f"Price = {row.get('w30', np.nan):.4f} × ASP30 + {row.get('w90', np.nan):.4f} × ASP90 + {row.get('wf', np.nan):.4f} × FamilyAdjustedPrice",
        'test_balanced_MAE': round(row['balanced_MAE'], 2),
        'test_unit_weighted_MAE': round(row['unit_weighted_MAE'], 2),
        'test_R2': round(row['R2'], 4),
    }
print(json.dumps(conclusion, indent=2))
with open(ARTIFACT_DIR / 'corrected_model_conclusion.json', 'w') as file:
    json.dump(conclusion, file, indent=2)

{
  "CA": {
    "selected_model": "ASP30_only",
    "formula": "Price = 1.0000 \u00d7 ASP30 + 0.0000 \u00d7 ASP90 + 0.0000 \u00d7 FamilyAdjustedPrice",
    "test_balanced_MAE": 333.41,
    "test_unit_weighted_MAE": 368.95,
    "test_R2": 0.2542
  },
  "US": {
    "selected_model": "ASP30_only",
    "formula": "Price = 1.0000 \u00d7 ASP30 + 0.0000 \u00d7 ASP90 + 0.0000 \u00d7 FamilyAdjustedPrice",
    "test_balanced_MAE": 255.11,
    "test_unit_weighted_MAE": 266.27,
    "test_R2": 0.6321
  }
}


## Limitations

- The model predicts a defensible historical price baseline, not a causal profit-maximizing price.
- Family peers are limited to the selected Top 30 products. A production model should calculate peer signals from the full eligible product universe.
- Products without sufficient 30-day/90-day or peer history are excluded. Production requires an explicit cold-start fallback.
- FY27Q1 contains relatively few eligible CA product-days; interpret CA test results with additional caution.
- Family, product-type, and product-specific weights should be introduced only after diagnostic and rolling-validation gates are passed.